# 02 â€” Feature Engineering (v2)

**Purpose:** Build the training dataset for v2 pipeline.

**Key changes from v1:**
- `last_7_day_avg`, `last_30_day_avg`, `last_60_day_avg` **removed** from features (they encode WMA directly â€” leakage)
- `product_id_encoded` (ordinal) replaced with **target encoding** (mean qty per product in training period)
- 5 new engineered features added
- Small sample (300 products) to avoid RAM crash in notebook

**Run after notebook 01.** Uses same DB connection pattern.

---

## Cell 1 â€” Load data (full, unfiltered)

**Important:** Load WITHOUT filtering stockout rows first. We need the full history to compute sequential features like `days_since_last_sale` and `zero_run_length`. Filtering happens after feature engineering.

In [ ]:
import os, gc
from pathlib import Path
from urllib.parse import quote_plus
from datetime import timedelta

import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text

load_dotenv(find_dotenv())
password = quote_plus(os.getenv("password", ""))
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('user')}:{password}"
    f"@{os.getenv('host')}:{os.getenv('port')}/{os.getenv('dbname')}",
    future=True, pool_pre_ping=True,
)

# Step 1: fetch product list only (cheap), then pick first 300
SAMPLE_N = 300
all_products = pd.read_sql(
    text("SELECT DISTINCT product_id FROM derived.product_daily_features ORDER BY product_id"),
    con=engine,
)["product_id"].tolist()
sample_products = all_products[:SAMPLE_N]

# Step 2: load ONLY those 300 products — ~392K rows instead of 2.9M
# This is the fix that prevents OOM on 16 GB RAM.
df = pd.read_sql(
    text("""
        SELECT date, product_id, quantity_sold,
               lag_1_qty, lag_7_qty,
               last_7_day_avg, last_30_day_avg, last_60_day_avg,
               last_7_day_stddev, day_of_week,
               is_holiday, days_to_next_festival, days_since_last_festival,
               stockout_proxy
        FROM derived.product_daily_features
        WHERE product_id = ANY(:pids)
        ORDER BY product_id, date
    """),
    con=engine,
    parse_dates=["date"],
    params={"pids": sample_products},
)
# Cast numerics to float32 immediately — halves memory vs float64
float_cols = [
    "quantity_sold", "lag_1_qty", "lag_7_qty",
    "last_7_day_avg", "last_30_day_avg", "last_60_day_avg", "last_7_day_stddev",
]
df[float_cols] = df[float_cols].astype("float32")

print(f"Loaded {len(df):,} rows, {df['product_id'].nunique():,} products "
      f"(full catalog: {len(all_products):,})")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## Cell 2 result — 300 products loaded directly from DB

Loading only the sample products at query time (not post-load filtering) keeps peak RAM under 200 MB.
`df` here is already the 300-product sample — `df_sample` below is just an alias.

Production `train.py` uses full 2.9M rows with chunked loading + float32 throughout.

In [ ]:
# df is already the 300-product sample — alias for compatibility with remaining cells
df_sample = df
print(f"df_sample: {len(df_sample):,} rows, {df_sample['product_id'].nunique():,} products")

## Cell 3 â€” Add 5 engineered features

These capture signals the rolling averages miss:

| Feature | Captures |
|---|---|
| `days_since_last_sale` | How stale is this product? |
| `rolling_non_zero_count` | Sale frequency in last 7 days |
| `demand_trend_ratio` | Rising vs falling trend (last_7 / last_30) |
| `zero_run_length` | Current dry spell length |
| `is_recently_active` | Binary: any sale in last 7 days? |

In [ ]:
df_sample = df_sample.sort_values(["product_id", "date"]).copy()

# 1. days_since_last_sale
df_sample["_sale_date"] = df_sample["date"].where(df_sample["quantity_sold"] > 0)
df_sample["_last_sale"] = df_sample.groupby("product_id")["_sale_date"].ffill()
df_sample["days_since_last_sale"] = (df_sample["date"] - df_sample["_last_sale"]).dt.days.fillna(999)
df_sample.drop(columns=["_sale_date", "_last_sale"], inplace=True)

# 2. rolling_non_zero_count
df_sample["rolling_non_zero_count"] = (
    df_sample.groupby("product_id")["quantity_sold"]
    .transform(lambda x: (x > 0).astype(float).rolling(7, min_periods=1).sum())
)

# 3. demand_trend_ratio  (last7 / last30 â€” captures direction, not level)
df_sample["demand_trend_ratio"] = (
    df_sample["last_7_day_avg"] / (df_sample["last_30_day_avg"] + 1e-6)
).clip(0, 10)

# 4. zero_run_length  (consecutive zeros ending at current row)
df_sample["_is_zero"] = df_sample["quantity_sold"] == 0
df_sample["_nz_cumsum"] = (~df_sample["_is_zero"]).groupby(df_sample["product_id"]).cumsum()
df_sample["zero_run_length"] = (
    df_sample["_is_zero"].astype(int)
    .groupby([df_sample["product_id"], df_sample["_nz_cumsum"]])
    .cumsum()
)
df_sample.drop(columns=["_is_zero", "_nz_cumsum"], inplace=True)

# 5. is_recently_active
df_sample["is_recently_active"] = (df_sample["rolling_non_zero_count"] > 0).astype(int)

print("New features added:")
print(df_sample[["date", "product_id", "quantity_sold",
                 "days_since_last_sale", "rolling_non_zero_count",
                 "demand_trend_ratio", "zero_run_length", "is_recently_active"]].tail(10))
gc.collect()
print(f"After feature engineering: "
      f"{df_sample.memory_usage(deep=True).sum()/1e6:.1f} MB")

## Cell 4 â€” Filter training rows

Now filter after feature engineering. We exclude:
- Stockout rows (biased zeros)
- Missing lag rows (first days per product)

In [ ]:
df_clean = df_sample[
    (~df_sample["stockout_proxy"]) &
    df_sample["lag_1_qty"].notna() &
    df_sample["lag_7_qty"].notna()
].copy().reset_index(drop=True)

print(f"Before filter: {len(df_sample):,}")
print(f"After filter:  {len(df_clean):,}  ({len(df_clean)/len(df_sample)*100:.1f}% retained)")

## Cell 5 â€” Target encoding for product_id

**Why not ordinal encoding?**
Ordinal encoding (`PROD001â†’0, PROD002â†’1, PROD003â†’2`) introduces a false numeric ordering. XGBoost's tree splits interpret product 500 as "twice" product 250 â€” meaningless.

**Target encoding** replaces each product_id with its mean `quantity_sold` in the training period. Products with similar demand get similar values. The model learns "this product typically sells X units" as a continuous signal.

**Critical:** compute the mean from TRAINING rows only. Using the full dataset (including test) would leak future information.

In [ ]:
TEST_DAYS = 30
cutoff = df_clean["date"].max() - pd.Timedelta(days=TEST_DAYS)

# Compute encoding from TRAIN rows only
train_rows = df_clean[df_clean["date"] <= cutoff]
product_mean = train_rows.groupby("product_id")["quantity_sold"].mean()
global_mean  = float(train_rows["quantity_sold"].mean())

df_clean["product_id_target_encoded"] = (
    df_clean["product_id"].map(product_mean).fillna(global_mean)
)

print(f"Train cutoff: {cutoff.date()}")
print(f"Global mean: {global_mean:.4f}")
print(f"\nSample product encodings (sorted by mean demand):")
print(product_mean.sort_values(ascending=False).head(10).reset_index()
      .rename(columns={"quantity_sold": "target_encoded_value"}))

## Cell 6 â€” Feature columns definition

**Note what's excluded:** `last_7_day_avg`, `last_30_day_avg`, `last_60_day_avg` â€” these directly feed the WMA formula. Training with them causes the model to learn a noisy version of WMA rather than learning new patterns.

`demand_trend_ratio` (derived from those columns) is OK â€” it captures direction of change, not the level.

In [ ]:
FEATURE_COLS = [
    # Lag features
    "lag_1_qty",
    "lag_7_qty",
    # Volatility
    "last_7_day_stddev",
    # Calendar
    "day_of_week",
    "is_holiday",
    "days_to_next_festival",
    "days_since_last_festival",
    # New engineered features
    "days_since_last_sale",
    "rolling_non_zero_count",
    "demand_trend_ratio",
    "zero_run_length",
    "is_recently_active",
    # Product identity (target encoded)
    "product_id_target_encoded",
]

print(f"{len(FEATURE_COLS)} features")
print("NULL check:")
print((df_clean[FEATURE_COLS].isnull().sum() / len(df_clean) * 100).round(2))

## Cell 7 â€” Build single-horizon dataset (h=1)

For notebooks we train h=1 only (most important horizon, proves the approach works).
Production `train.py` trains h=1..7 separately.

**No 7x memory multiplication** â€” this is why separate models per horizon is better.
Each model sees ~(N rows Ã— 13 features) at a time.

In [ ]:
HORIZON = 1

df_sorted = df_clean.sort_values(["product_id", "date"])
target = df_sorted.groupby("product_id")["quantity_sold"].shift(-HORIZON)
valid = target.notna()

X = df_sorted.loc[valid, FEATURE_COLS].fillna(0).astype("float32")
y = target[valid].astype("float32")
dates = df_sorted.loc[valid, "date"]

# Keep these for WMA fair comparison
wma_inputs = df_sorted.loc[valid, ["last_7_day_avg", "last_30_day_avg", "last_60_day_avg"]]

train_mask = dates <= cutoff
test_mask  = dates > cutoff

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]
wma_test_inputs  = wma_inputs[test_mask]

print(f"h={HORIZON} â€” Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Train date range: {dates[train_mask].min().date()} â†’ {dates[train_mask].max().date()}")
print(f"Test  date range: {dates[test_mask].min().date()}  â†’ {dates[test_mask].max().date()}")